# Feature Engineering

In [ ]:
# Make sure you are at the parent directory
from pathlib import Path
import sys

# Define MODE
MODE = "EXPANSE" # Either "COLAB", "LOCAL", "EXPANSE"
if MODE.upper() not in ('EXPANSE', 'COLAB', 'LOCAL'):
    raise Exception("Invalid mode, the only acceptible are 'EXPANSE', 'COLAB', 'LOCAL'")
if MODE.upper() == "COLAB":
    from google.colab import drive
    drive.mount('/content/drive')

# Root path by MODE
PROJECT_ROOT_BY_MODE = {
    "COLAB": Path("/content/drive/MyDrive/DSC 288R/Project"),
    "EXPANSE": Path("/home/bguo3/bguo3/DSC-288R-Capstone-Final-Project"),
    "LOCAL": Path("/Users/steveg/Desktop/DSC-288R-Capstone-Final-Project"),
}
ROOT = PROJECT_ROOT_BY_MODE[MODE.upper()]

# Add the root path to global system
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT.resolve()))

# Print the root path
print("MODE:", MODE)
print("PROJECT_ROOT:", ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MODE: COLAB
PROJECT_ROOT: /content/drive/MyDrive/DSC 288R/Project


## Import Modules

In [2]:
from pyspark.sql import functions as F

from src.utils.pyspark_utils import create_spark_session, memory_count
from src.utils.paths_utils import ProjectPaths
from src.utils.io_utils import (
    read_spark_parquet,
    write_spark_parquet,
    write_pandas_parquet
)
import src.pipelines.feature_engineering as feat

## Read Cleaned Full/Sampled Dataset From Parquet

In [3]:
# Set up for Spark app & resource allocation
spark = create_spark_session("steam_reviews_machine_learning_modeling")

# Load data
paths = ProjectPaths(MODE)
if MODE.upper() == "EXPANSE":
    df = read_spark_parquet(spark=spark, path=paths.cleaned_parquet)
else:
    df = read_spark_parquet(spark=spark, path=paths.cleaned_sampled_parquet)

Read Spark parquet from: /content/drive/MyDrive/DSC 288R/Project/data/cleaned_sampled.parquet


## Select Relevant Features

In [4]:
selected_cols = [
    # User profile / activity counts
    "author_num_games_owned",
    "author_num_reviews",

    # Playtime engagement
    "author_playtime_forever",
    "author_playtime_at_review",
    "author_playtime_last_two_weeks",

    # Temporal / recency inputs
    "author_last_played",
    "timestamp_created",
    "timestamp_updated",

    # Review behavior / sentiment proxy
    "voted_up",

    # Review engagement
    "votes_up",
    "weighted_vote_score"
]
df = df.transform(feat.reduce_features, cols=selected_cols, strict=True)
df.printSchema()

root
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_playtime_last_two_weeks: integer (nullable = true)
 |-- author_last_played: long (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- weighted_vote_score: float (nullable = true)



## Create Label for Supervised Machine Learning Tasks

For this initial baseline model, a player is considered **churned** if they recorded less than 60 minutes of playtime within the last two weeks. Players with 60 minutes or more are treated as likely **NOT churned** users.

The 60-minute threshold acts as apractical behavioral heuristic rather than a definitive business rule. The assumption is that players who return and spend at least one hour actively engaging with a game over a recent two-week period demonstrate meaningful continued interest and engagement. In contrast, very low or zero recent playtime may indicate disengagement, abandonment, or temporary inactivity.

This threshold was intentionally chosen as a lightweight and interpretable starting point for experimentation. It helps transform continuous playtime behavior into a binary classification problem suitable for Logistic Regression while remaining easy to explain from a business perspective.

**FUTURE ITERATION** of the project may refine this definition using:
- percentile-based engagement thresholds,
- genre-specific activity expectations,
- rolling activity windows,
- survival analysis,
- or clustering methods to identify natural retention breakpoints.

In [5]:
df = df.transform(feat.create_label)
df.printSchema()

root
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: long (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- weighted_vote_score: float (nullable = true)
 |-- churn: double (nullable = false)



## Handle Class Imbalance by Adding Class Weight Column

In [6]:
df = df.transform(feat.create_class_weights)
df.printSchema()

root
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: long (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- weighted_vote_score: float (nullable = true)
 |-- churn: double (nullable = false)
 |-- class_weight: double (nullable = true)



## Derive Review Behavior Features

In [7]:
df = df.transform(feat.create_review_behavior_features)
df.printSchema()

root
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: long (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- weighted_vote_score: float (nullable = true)
 |-- churn: double (nullable = false)
 |-- class_weight: double (nullable = true)
 |-- review_positive: integer (nullable = true)



## Derive Engagement Features

In [8]:
# df = df.transform(feat.create_engagement_features)
# df.printSchema()

## Derive Log Transformed Features

In [9]:
df = df.transform(feat.transform_log_features, drop_original=True)
df.printSchema()

root
 |-- author_last_played: long (nullable = true)
 |-- timestamp_created: long (nullable = true)
 |-- timestamp_updated: long (nullable = true)
 |-- voted_up: boolean (nullable = true)
 |-- churn: double (nullable = false)
 |-- class_weight: double (nullable = true)
 |-- review_positive: integer (nullable = true)
 |-- log_author_num_games_owned: double (nullable = true)
 |-- log_author_num_reviews: double (nullable = true)
 |-- log_author_playtime_forever: double (nullable = true)
 |-- log_author_playtime_at_review: double (nullable = true)
 |-- log_votes_up: double (nullable = true)
 |-- log_weighted_vote_score: double (nullable = true)



## Assemble Feature, Labels & Class Weights for Machine Learning

In [10]:
assembled_features = [col for col in df.columns if col not in [feat.LABEL_COL, feat.WEIGHT_COL]]
df = feat.assemble_features(df, feature_cols=assembled_features, handle_invalid='error', strict=False)
df.printSchema()

['author_last_played', 'timestamp_created', 'timestamp_updated', 'voted_up', 'review_positive', 'log_author_num_games_owned', 'log_author_num_reviews', 'log_author_playtime_forever', 'log_author_playtime_at_review', 'log_votes_up', 'log_weighted_vote_score']
{'log_author_playtime_at_review', 'class_weight', 'timestamp_updated', 'log_votes_up', 'churn', 'log_weighted_vote_score', 'timestamp_created', 'voted_up', 'log_author_num_games_owned', 'author_last_played', 'log_author_num_reviews', 'review_positive', 'log_author_playtime_forever'}
[]
['author_last_played', 'timestamp_created', 'timestamp_updated', 'voted_up', 'review_positive', 'log_author_num_games_owned', 'log_author_num_reviews', 'log_author_playtime_forever', 'log_author_playtime_at_review', 'log_votes_up', 'log_weighted_vote_score']
root
 |-- author_last_played: double (nullable = true)
 |-- timestamp_created: double (nullable = true)
 |-- timestamp_updated: double (nullable = true)
 |-- voted_up: double (nullable = true)
 |

## Finalized the Columns for Machine Learning

In [11]:
selected_cols = [
    "finalized_features",
    "churn",
    "class_weight",
]
df = df.transform(feat.reduce_features, cols=selected_cols, strict=True)
df.printSchema()

root
 |-- finalized_features: vector (nullable = true)
 |-- churn: double (nullable = false)
 |-- class_weight: double (nullable = true)



## Write Feature Engineered Data to Parquet File

In [12]:
if MODE == "EXPANSE":
    write_spark_parquet(df, paths.feature_engineered_parquet)
else:
    write_spark_parquet(df, paths.feature_engineered_sampled_parquet)


Saved Spark parquet to: /content/drive/MyDrive/DSC 288R/Project/data/feature_engineered_sampled.parquet


## Future Development

In [13]:
# FUTURE DEVELOPMENT

# # text canonicalization
# lowercasing
# URL replacement
# emoji handling
# punctuation normalization
# accent normalization
# contraction handling
# tokenization
# stopword removal
# stemming / lemmatization
# language-specific processing

# # text vectorization & feature engineering
# TF-IDF
# word n-grams
# character n-grams
# sentiment scores
# embeddings
# language-specific vectorizers
# review length features